# vLLM 실제로 돌려보기 (Colab)

지금까지는 커널 하나만 따로 떼어서 봤다면, 이번에는 vLLM을 통째로 설치하고 작은 모델을 직접 돌려서, "요청 → 스케줄러 → attention 커널"이라는 전체 파이프라인이 로그로 어떻게 찍히는지 직접 확인해봅니다.

**런타임을 GPU로 설정하세요** (메뉴 → 런타임 → 런타임 유형 변경 → GPU). 무료 T4는 VRAM 16GB라 작은 모델(1~2B급)만 돌릴 수 있습니다.

**설치가 약간 걸립니다** (보통 3~6분) — 의존성이 많아서 그렇습니다.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q vllm
# vllm이 새 PyTorch(CUDA 버전 다름)를 깔면서, Colab에 미리 깔려있던 torchaudio와
# CUDA 버전이 어긋나 import 시점에 RuntimeError가 납니다. torchaudio는 안 쓰니
# 지웁니다. torchvision은 vllm이 내부적으로(멀티모달 프로세서 모듈 import 시)
# 요구해서 지우면 안 됩니다 — 남겨둡니다.
!pip uninstall -y -q torchaudio

## 0.5 GPU 인식 확인 (LLM() 만들기 전에)

vLLM은 GPU가 있는지를 `pynvml`(NVML) 초기화로 확인하는데, 이게 실패하면 vLLM이 그 원인을 완전히 삼켜버리고
그냥 "플랫폼 감지 실패"로 넘어가서 나중에 `torch.device("")`처럼 뜬금없는 에러로 터집니다.
그래서 `LLM(...)`을 만들기 전에 이 확인부터 먼저 해서 진짜 원인을 봅니다.

In [ ]:
import torch

print("torch:", torch.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("torch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device name:", torch.cuda.get_device_name(0))

print()
try:
    from vllm.utils.import_utils import import_pynvml

    pynvml = import_pynvml()
    pynvml.nvmlInit()
    try:
        count = pynvml.nvmlDeviceGetCount()
        print("pynvml OK, device count:", count)
    finally:
        pynvml.nvmlShutdown()
except Exception as e:
    print("pynvml/NVML 초기화 실패 (vLLM이 이걸 조용히 삼켜서 위에서는 안 보였던 진짜 원인):")
    print(f"  {type(e).__name__}: {e}")

## 1. 작은 모델 로드

`TinyLlama-1.1B-Chat`을 씁니다 — 우리가 소스코드로 추적했던 `LlamaAttention` 경로를 그대로 타는 Llama 아키텍처 모델이고, 게이트 없는 라이선스라 바로 다운로드됩니다.

vLLM이 모델을 초기화하는 동안 나오는 로그가 많은데, 그 안에 우리가 지난번에 코드로 봤던 "어떤 attention 백엔드를 고르는지", "KV 캐시를 어떻게 잡는지" 같은 줄들이 있습니다. `profiler_config`로 나중에(4번) 쓸 프로파일러 출력 경로도 미리 지정해둡니다.

In [ ]:
import ctypes
import os

# 엔진 코어를 별도 서브프로세스로 안 띄우고 이 노트북 프로세스 안에서 그대로
# 돌립니다. Jupyter/Colab처럼 이미 CUDA가 초기화된 프로세스에서 fork로 서브
# 프로세스를 새로 띄우면 깨지는 경우가 많아서, 이 값을 꺼서 우리가 지난번에
# 코드로 봤던 InprocClient 경로를 강제로 타게 합니다.
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

# vLLM은 GPU 워커 초기화(NCCL 프로세스 그룹 생성) 때 stdout을 잠깐 숨기려고
# sys.stdout.fileno()를 호출하는데, Jupyter/Colab의 stdout은 ipykernel이 만든
# 가짜 스트림이라 fileno()가 없어서 `io.UnsupportedOperation: fileno`로 죽습니다.
# vllm/utils/system_utils.py의 suppress_stdout()이 VLLM_LOGGING_LEVEL=DEBUG일
# 때는 이 fileno() 호출 자체를 건너뛰도록 이미 만들어져 있어서, 이걸로 우회합니다.
os.environ["VLLM_LOGGING_LEVEL"] = "DEBUG"

# vllm이 설치한 torch가 요구하는 libcudart.so.13이 .../nvidia/cu13/lib/ 밑에
# 있는데 LD_LIBRARY_PATH엔 안 잡혀 있어서 "libcudart.so.13: cannot open
# shared object file"로 죽습니다. os.environ["LD_LIBRARY_PATH"]를 바꾸는 건
# 이미 떠 있는 프로세스엔 반영이 안 되므로(동적 링커가 시작 시점에 캐싱),
# 절대경로로 직접 미리 로드해서 나중에 재사용되게 합니다.
_cu13_libcudart = (
    "/usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib/libcudart.so.13"
)
if os.path.isfile(_cu13_libcudart):
    ctypes.CDLL(_cu13_libcudart, mode=ctypes.RTLD_GLOBAL)

from vllm import LLM, SamplingParams

os.makedirs("/content/vllm_trace", exist_ok=True)

llm = LLM(
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    gpu_memory_utilization=0.85,
    max_model_len=2048,
    # 4번 스텝의 프로파일러가 여기서 만든 트레이스 폴더를 씁니다.
    profiler_config={"profiler": "torch", "torch_profiler_dir": "/content/vllm_trace"},
)

## 2. 위 출력에서 attention 백엔드 찾기

바로 위 셀을 실행한 로그 전체를 위로 스크롤해서 `backend`나 `Attn`이 들어간 줄을 찾아보세요. 우리가 `vllm/v1/attention/backends/registry.py`에서 봤던 `TRITON_ATTN`, `FLASH_ATTN` 같은 이름이 그대로 보일 겁니다. (Colab 무료 T4는 구세대 아키텍처라 FlashAttention이 안 되는 경우가 많고, 그럴 때 vLLM이 자동으로 Triton 백엔드로 폴백합니다 — 우리가 지난번에 직접 뜯어본 바로 그 커널입니다.)

In [ ]:
# 엔진이 실제 사용 중인 attention backend를 코드로도 확인
attn_layer = None
for name, module in llm.llm_engine.model_executor.driver_worker.model_runner.model.named_modules():
    cls_name = type(module).__name__
    if cls_name == "Attention":
        attn_layer = module
        break

if attn_layer is not None:
    print("attn backend:", attn_layer.attn_backend.get_name())
    print("impl class  :", type(attn_layer.impl).__name__)
else:
    print("Attention 레이어를 모델 모듈 트리에서 못 찾았습니다 (vLLM 버전마다 내부 경로가 다를 수 있습니다).")

이 `attn_backend.get_name()`과 `type(attn_layer.impl).__name__`이 우리가 지난번에 코드로 따라갔던 경로와 정확히 일치해야 합니다:

```
LlamaAttention.forward()
  → self.attn (= Attention 레이어, 바로 위에서 찾은 것)
    → self.attn_backend            (예: TritonAttentionBackend)
      → self.impl                  (예: TritonAttentionImpl)
        → unified_attention(...)   (@triton.jit 실제 커널)
```

## 3. 실제 생성해보기

In [ ]:
prompts = [
    "The capital of France is",
    "def fibonacci(n):",
]
sampling_params = SamplingParams(temperature=0.0, max_tokens=32)

outputs = llm.generate(prompts, sampling_params)
for out in outputs:
    print("PROMPT:", out.prompt)
    print("OUTPUT:", out.outputs[0].text)
    print("---")

## 4. 스케줄러/스텝 단위로 들여다보기 (지난번 추적한 파이프라인 확인)

우리가 코드로 따라갔던 `EngineCore.step()`은 매 스텝마다:
1. `scheduler.schedule()` — 이번 배치에 넣을 요청/토큰 결정
2. `model_executor.execute_model()` — 실제 GPU forward
3. `sample_tokens()` — 다음 토큰 샘플링

이걸 직접 들여다보려면 vLLM의 native profiler(`start_profile`/`stop_profile`)를 쓰면 Chrome trace를 만들 수 있습니다. (파일이 꽤 커서 Colab에서 다운로드해 열어봐야 합니다. 최신 Chrome은 `chrome://tracing`을 막아둔 경우가 많아서, 대신 [ui.perfetto.dev](https://ui.perfetto.dev)를 씁니다 — 같은 트레이스 포맷을 그대로 지원합니다.)

In [ ]:
# torch_profiler_dir은 3번(LLM 생성) 시점에 이미 지정해뒀으므로 여기선 켜고 끄기만 하면 됩니다.
llm.start_profile()
_ = llm.generate(
    ["Explain what a KV cache is in one sentence."],
    SamplingParams(temperature=0.0, max_tokens=32),
)
llm.stop_profile()

print(os.listdir("/content/vllm_trace"))

`/content/vllm_trace`에 `.json` 트레이스 파일이 생깁니다. 다운로드해서 [ui.perfetto.dev](https://ui.perfetto.dev)에서 "Open trace file"로 열거나 드래그해보면, 타임라인 위에 스케줄러/model forward/샘플러가 각각 얼마나 시간을 먹는지 시각적으로 보입니다 — 우리가 코드로만 보던 파이프라인이 실제 시간으로 어디에 쓰이는지 확인하는 거예요.

In [ ]:
from google.colab import files
import glob

trace_files = glob.glob("/content/vllm_trace/*.json")
if trace_files:
    files.download(trace_files[0])
else:
    print("trace 파일이 아직 없습니다 — 위 셀이 에러 없이 끝났는지 확인하세요.")

## 5. 실제 프로파일링 결과: 병목은 attention이 아니라 GEMV

`torch.profiler`로 4번 셀의 생성 한 번을 찍어보니(`CUDA total` 기준 정렬), 실제 GPU 시간 분포는 이랬습니다:

| 순위 | 커널 | CUDA 시간 | 비중 | 정체 |
|---|---|---|---|---|
| 1 | `internal::gemvx...` (cuBLAS, 2개) | 188.9ms + 50.8ms | **86.8%** | GEMV(행렬-벡터곱) — QKV/O-proj/MLP 같은 linear layer |
| 2 | `aten::mm` | 15.4ms | 5.6% | 배치된 행렬곱 (prefill 단계) |
| 3 | `cutlass...wmma_tensorop` | 11.5ms | 4.2% | 텐서코어 GEMM |
| 4 | `kernel_unified_attention` | 6.7ms | **2.4%** | 우리가 소스코드로 뜯어봤던 그 Triton attention 커널 |
| 5 | `triton_red_fused_..._rms_norm` | ~6.6ms | ~2.4% | RMSNorm (torch.compile이 자동 fuse) |
| 6 | `reshape_and_cache_kernel_flash` | 1.9ms | 0.7% | PagedAttention KV 캐시 write |

**핵심 발견**: attention 커널은 전체 GPU 시간의 2.4%밖에 안 됩니다. 압도적 1위는 **GEMV**예요.

### 왜 GEMV가 병목인가

디코딩은 토큰을 한 번에 하나씩 생성합니다(배치 1). 행렬곱 `(M, K) x (K, N)`에서 `M`(배치 크기)이 크면 가중치 하나를 여러 번 재사용하지만(연산 대비 메모리 접근이 적음 = compute-bound), `M=1`이면 가중치 원소 하나당 곱셈을 딱 한 번만 하고 버립니다 — 즉 **연산량 대비 "가중치를 얼마나 빨리 HBM에서 읽어오느냐"가 전부를 결정**합니다(memory-bandwidth-bound). GPU의 막강한 연산 능력이 거의 낭비되는 상황이에요.

### 어떻게 해결하나

1. **배치 키우기 (continuous batching)** — 여러 요청을 동시에 디코딩하면 GEMV가 사실상 GEMM이 돼서, 가중치 하나를 여러 요청이 나눠 씁니다. vLLM이 존재하는 핵심 이유가 바로 이거예요. (지금 이 노트북은 프롬프트 1개로만 돌려서 이 이점이 전혀 안 보인 것)
2. **가중치 양자화 (fp16 → int8/int4)** — HBM에서 읽어야 할 바이트 수 자체를 줄입니다. GEMV가 메모리 대역폭 병목이라는 게 핵심이니, **바이트 수를 줄이는 게 배치를 못 키우는 상황(배치=1)에서도 직접 먹히는 유일한 해법**입니다. 이게 바로 우리가 [group_quant_walkthrough.ipynb](group_quant_walkthrough.ipynb)에서 만들어본 커널이 실제로 푸는 문제예요 — 이제 그 커널이 "왜 필요한지"를 이 프로파일링 결과가 숫자로 설명해줍니다.
3. **Speculative decoding** — 작은 draft 모델이 토큰 여러 개를 미리 제안하고, 큰 모델이 한 번의 배치된 forward로 검증 — 여러 디코딩 스텝의 가중치 읽기를 한 번으로 합칩니다.
4. **커널 레벨 최적화** — cuBLAS의 범용 GEMV 대신, dequant+GEMV를 하나로 합친 커스텀 커널(양자화된 가중치를 읽으면서 그 자리에서 바로 GEMV) 같은 걸 짜면 "압축된 가중치 읽기"와 "행렬-벡터곱"을 커널 하나로 fuse해서 메모리 왕복을 더 줄일 수 있습니다 — 정확히 이런 종류의 작업이 HyperAccel 같은 edge 추론 가속기 회사가 하는 일입니다.